# 01 · Quickstart — capture, episode, summary, PR notes

Episodic turns a coding-agent session into one structured **CodingEpisode**. This notebook uses the
built-in synthetic factory (`episodic.testing`) so it runs with **no agent, no network, no models**.

In [1]:
from episodic.testing import make_episode
from episodic.schema import validate_episode

ep = make_episode("ep_demo_1", intent="add retry to the http client",
                  outcome="merged", feedback=["useful"], passed=3, failed=0)
print("schema errors:", validate_episode(ep))
print("intent:", ep["intent"])
print("steps:", [s["type"] for s in ep["steps"]])
print("tests ok:", ep["tests"][0]["ok"], "| composite reward:", ep["reward_vector"]["composite"])

schema errors: []
intent: add retry to the http client
steps: ['user_prompt', 'file_edit', 'shell_command']
tests ok: True | composite reward: 0.875


Each step is an `(action -> observation)` pair — the raw material for everything downstream.

In [2]:
for s in ep["steps"]:
    print(f"[{s['type']:<14}] {str(s['input'])[:50]:<52} -> {s['observation'][:40]!r}")

[user_prompt   ] {'prompt': 'add retry to the http client'}           -> ''
[file_edit     ] {'file_path': 'src/http.py'}                         -> 'applied edit to src/http.py'
[shell_command ] {'command': 'pytest -q'}                             -> '============================= test sessi'


Heuristic summary + suggested PR notes — useful **before any ML exists**.

In [3]:
from episodic.core import summary as summary_mod
report = summary_mod.summarize(ep)
print(summary_mod.render_markdown(report))

# Session summary — ep_demo_1

**Intent:** add retry to the http client

## What changed
- modified: src/http.py +12/-2

## Why it changed
add retry to the http client

## Files touched
- src/http.py

## Tests run
- pytest: 3 passed / 0 failed [ok] (pytest -q)

## Tests missing
- No test files changed for 1 edited source file(s): src/http.py

## Risky edits
_None detected._

## Suggested PR title
Add retry to the http client

## Suggested PR description
## Summary
add retry to the http client

## Changes
- modified: src/http.py +12/-2

## Tests
- pytest: 3 passed / 0 failed [ok] (pytest -q)

## Risks
_None._

## Follow-ups
- [ ] Add tests covering the changed source files.


## Follow-up TODOs
- Add tests covering the changed source files.


Feedback is a reward signal. A bad outcome scores lower than a merged one:

In [4]:
good = make_episode("ep_good", outcome="merged", feedback=["useful"], passed=3, failed=0)
bad = make_episode("ep_bad", outcome="reverted", feedback=["wrong"], passed=1, failed=2)
print("merged+useful :", good["reward_vector"]["composite"])
print("reverted+wrong:", bad["reward_vector"]["composite"])
assert good["reward_vector"]["composite"] > bad["reward_vector"]["composite"]
print("OK")

merged+useful : 0.875
reverted+wrong: 0.075
OK
